# Set up

Download August 2026 data to then load it to the database
```bash
# sync the libraries installed
pipenv shell
pipenv install -d Pipfile # to install all missing dependencies

# Data from August 2026 
scraper update 2026-08-01 2026-08-31
```

In [125]:
import ast
import sqlite3
from datetime import datetime
from pathlib import Path

import pandas as pd
import ollama
from tqdm import tqdm

from un_llm_analysis.extractor import extract_briefing, ExtractedSection, ExtractedBriefing
from un_llm_analysis.config import RAW_DATA_DIR, BRIEFINGS_DB, SCHEMA_DB

In [ ]:
# This cell won't be necessary with the database interface, but this is for now ad-hoc insert functions

with open(SCHEMA_DB, "r", encoding="utf-8") as f:
    sql_schema = f.read()

with sqlite3.connect(BRIEFINGS_DB) as conn:
    conn.executescript(sql_schema)


def insert_into_briefing(briefing: ExtractedBriefing) -> None:
    query_insert = """INSERT INTO briefing
            (briefing_id, briefing_date, url, status, http_status, error, speaker_name, speaker_role, full_text, downloaded_at, processed_at)
            VALUES(?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?);"""
    query_delete = """delete from briefing where briefing_id = ?;"""
    with sqlite3.connect(BRIEFINGS_DB) as conn:
        cursor = conn.cursor()
        cursor.execute(
            query_insert,
            (
                get_briefing_id_from_date(briefing.briefing_date),
                briefing.briefing_date,
                briefing.url,
                briefing.status,
                briefing.http_status,
                briefing.error,
                briefing.speaker_name,
                briefing.speaker_role,
                briefing.full_text or "",
                briefing.downloaded_at,
                str(datetime.now()),
            ),
        )
        conn.commit()
        cursor.close()


def inser_into_section(briefing_id: int, sections: tuple[ExtractedSection, ...]) -> None:
    query_insert = """INSERT INTO "section"
                (section_id, briefing_id, section_order, title, "text")
                VALUES(?, ?, ?, ?, ?);"""
    query_delete = """delete from section where briefing_id = ?;"""
    data = []
    for i, section in enumerate(sections, start=1):
        data.append((briefing_id * 100 + i, briefing_id, i, section.title, section.text))

    with sqlite3.connect(BRIEFINGS_DB) as conn:
        cursor = conn.cursor()
        cursor.executemany(query_insert, data)
        conn.commit()
        cursor.close()


def get_section_data(start_date: str, end_date: str) -> pd.DataFrame:
    query = """select section_id, briefing_id, section_order, title, "text" from "section" s where briefing_id between ? and ?;"""
    with sqlite3.connect(BRIEFINGS_DB) as conn:
        cursor = conn.cursor()
        cursor.execute(
            query,
            (
                get_briefing_id_from_date(start_date),
                get_briefing_id_from_date(end_date),
            ),
        )
        rows = cursor.fetchall()
        column_names = [x[0] for x in cursor.description]
        cursor.close()
    df = pd.DataFrame(rows, columns=column_names)
    return df

In [ ]:
# These functions need to be included to the library
def insert_sdg(
    briefing_id: int, section_id: int, sdgs: list[str], model: str, created_at: str
) -> None:
    parsed_sdgs = ast.literal_eval(sdgs)
    if parsed_sdgs is None:
        return
    sql_insert = """INSERT INTO sdg
                    (briefing_id, section_id, sdg, model, created_at)
                    VALUES(?, ?, ?, ?, ?);"""
    with sqlite3.connect(BRIEFINGS_DB) as conn:
        cursor = conn.cursor()
        for sdg in parsed_sdgs:
            cursor.execute(sql_insert, (briefing_id, section_id, sdg, model, created_at))
        conn.commit()
        cursor.close()


def query_sdg() -> pd.DataFrame:
    sql_query = """select briefing_id, section_id, sdg, model, created_at from sdg;"""
    with sqlite3.connect(BRIEFINGS_DB) as conn:
        cursor = conn.cursor()
        cursor.execute(sql_query)
        rows = cursor.fetchall()
        column_names = [x[0] for x in cursor.description]
        cursor.close()
    df = pd.DataFrame(rows, columns=column_names)
    return df


def get_briefing_id_from_date(date: str) -> int:
    """Returns a YYYYMMDD int from a 'YYYY-MM-DD' str.

    Args:
        date (str): Date of the briefing. Format 'YYYY-MM-DD'.

    Returns:
        int: The briefing date in numeric format YYYYMMDD.
    """
    briefing_date = datetime.strptime(date, "%Y-%m-%d")
    return briefing_date.year * 10000 + briefing_date.month * 100 + briefing_date.day

In [ ]:
# Load data to the database for this notebook
# path = RAW_DATA_DIR / "briefings" / "2026" / "2026-08"
path = RAW_DATA_DIR / "briefings"
files = list(path.glob("**/*.json"))
for file_path in tqdm(files, total=len(files)):
    briefing = extract_briefing(file_path=file_path)
    insert_into_briefing(briefing=briefing)
    inser_into_section(
        briefing_id=get_briefing_id_from_date(briefing.briefing_date),
        sections=briefing.sections,
    )

100%|██████████| 407/407 [00:05<00:00, 78.65it/s]


# SDGs

In [81]:
# https://sdgs.un.org/goals
sdgs = {
    "SDG1": "End poverty in all its forms everywhere",
    "SDG2": "End hunger, achieve food security and improved nutrition and promote sustainable agriculture",
    "SDG3": "Ensure healthy lives and promote well-being for all at all ages",
    "SDG4": "Ensure inclusive and equitable quality education and promote lifelong learning opportunities for all",
    "SDG5": "Achieve gender equality and empower all women and girls",
    "SDG6": "Ensure availability and sustainable management of water and sanitation for all",
    "SDG7": "Ensure access to affordable, reliable, sustainable and modern energy for all",
    "SDG8": "Promote sustained, inclusive and sustainable economic growth, full and productive employment and decent work for all",
    "SDG9": "Build resilient infrastructure, promote inclusive and sustainable industrialization and foster innovation",
    "SDG10": "Reduce inequality within and among countries",
    "SDG11": "Make cities and human settlements inclusive, safe, resilient and sustainable",
    "SDG12": "Ensure sustainable consumption and production patterns",
    "SDG13": "Take urgent action to combat climate change and its impacts",
    "SDG14": "Conserve and sustainably use the oceans, seas and marine resources for sustainable development",
    "SDG15": "Protect, restore and promote sustainable use of terrestrial ecosystems, sustainably manage forests, combat desertification, and halt and reverse land degradation and halt biodiversity loss",
    "SDG16": "Promote peaceful and inclusive societies for sustainable development, provide access to justice for all and build effective, accountable and inclusive institutions at all levels",
    "SDG17": "Strengthen the means of implementation and revitalize the Global Partnership for Sustainable Development",
}

sdg_text = ""
for sdg in sdgs:
    sdg_text += f"* {sdg} {sdgs[sdg]}\n"

sdg_definitions = sdg_text.strip()

In [82]:
promtp = """**Classification Task:**

Classify the following text into one or more of the following 17 Sustainable Development Goals (SDGs):

**Text to be Classified:**

{text}

**SDG Definitions:**

{sdg_definitions}

**Example Response:**

If the text belongs to one or more of the 17 SDGs, provide a list of relevant goal(s) in the format: ['SDG1', 'SDG5'].

**Edge Case:**

If the text does not belong to any of the 17 SDGs, return 'None'. Do not include any text to your answer, just provide the response in a list.
"""

In [83]:
df_sections = get_section_data()
llm = "llama3.2"
for i, section in tqdm(df_sections.iterrows(), total=df_sections.shape[0]):
    result = ollama.generate(
        model=llm,
        prompt=promtp.format(text=section["text"], sdg_definitions=sdg_definitions),
        think=False,
    )

    insert_sdg(
        briefing_id=section.briefing_id,
        section_id=section.section_id,
        sdgs=result["response"],
        model=llm,
        created_at=result["created_at"],
    )

100%|██████████| 204/204 [16:19<00:00,  4.80s/it]


# Visualisation

In [ ]:
import plotly.express as px

df = query_sdg()
x = df[df["model"] == llm][["sdg"]].value_counts().values  # frequency of each SDG in the sections.
y = [
    x[0] for x in df[df["model"] == llm][["sdg"]].value_counts().index
]  # values from the "sdg" column with values: ["SDG1", "SDG2", ..., "SDG17"]

fig = px.bar(
    df,
    x=x,
    y=y,
    orientation="h",
    title=f"17SDGs in {len(df.briefing_id.unique())} Daily Briefings between {str(datetime.strptime(str(df.briefing_id.min()), "%Y%m%d"))[:10]} and {str(datetime.strptime(str(df.briefing_id.max()), "%Y%m%d"))[:10]}",
)
fig.show()

# mlflow (optional)

This is to compare results from different LLMs.

Install the following libraries and dependencies:
```bash
# Download LLMs to test
ollama pull llama3.2
ollama pull gemma3:4b

# Install dependencies to use mlflow with local Ollama
pipenv shell
pipenv install mlflow openai jupyter ipywidgets # or `pipenv install -d Pipfile to sync dependencies`

# Start a local mlflow server
mlflow server
```

In [ ]:
import mlflow
from openai import OpenAI

mlflow.openai.autolog()
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("SDG")

client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="dummy",
)

In [ ]:
models = ["llama3.2:latest", "gemma3:4b"]
for llm in models:
    for i, section in enumerate(df_sections.text.to_list()[:5]):
        response = client.chat.completions.create(
            model=llm,
            messages=[
                {
                    "role": "user",
                    "content": promtp.format(text=section, sdg_definitions=sdg_definitions),
                },
            ],
            temperature=0,
        )